# 05 — Legs & Formation Classification

## Why legs?

A valid Supply & Demand zone is not just a tight cluster — it is a **balanced structure with a price memory**.
The leg-in carries the energy that *created* the base; the leg-out is the burst that *leaves* it.
Without measuring both we cannot tell whether a cluster was formed by institutional intent or by random noise.

## The four formations

| Leg-in | Leg-out | Code | Zone type | Pattern name |
|--------|---------|------|-----------|--------------|
| Up     | Up      | RBR  | Demand    | Rally-Base-Rally |
| Down   | Down    | DBD  | Supply    | Drop-Base-Drop |
| Down   | Up      | DBR  | Demand    | Drop-Base-Rally |
| Up     | Down    | RBD  | Supply    | Rally-Base-Drop |

If either leg is **flat** the cluster is discarded — both momentum signals must be present.

## Leg-strength formula

Given cluster `[bs, be]` and window size $L$ = `LEG_CANDLES`:

$$
\text{leg\_in\_net}  = \text{close}_{bs-1} - \text{open}_{bs-L}
$$

$$
\text{leg\_out\_net} = \text{close}_{be+L} - \text{close}_{be}
$$

A leg is **directional** if $|\text{net}| \ge \text{LEG\_ATR\_MIN} \times \overline{\text{ATR}}_{\text{base}}$, otherwise it is **flat**.

| Constant | Default | Meaning |
|---|---|---|
| `LEG_CANDLES` | 3 | Bars to look back/forward on each side |
| `LEG_ATR_MIN` | 1.5 | Minimum ATR-multiples for a directional leg |

## 1. Setup & load data

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from IPython.display import display

from utils.data_loader import load_all_timeframes
from utils.models import CandlePrimitives, add_atr
from utils.base_detector import detect_bases
from utils.legs_formation import detect_formations, FORMATION_MAP
from utils.config import (
    COLOR_BULL, COLOR_BEAR, COLOR_BASE,
    CHART_BG as BG, CHART_GRID as GRID
)

SYMBOL = "USDJPY=X"

raw = load_all_timeframes(SYMBOL, align=False)
data = {tf: add_atr(CandlePrimitives.enrich_dataframe(df)) for tf, df in raw.items()}
common_start = max(df.index[0] for df in raw.values())
data = {tf: df[df.index >= common_start] for tf, df in data.items()}

print(f"Loaded {len(data)} timeframes: {list(data.keys())}")
for tf, df in data.items():
    print(f"  {tf:4s}  {len(df):5d} rows  {df.index[0].date()} → {df.index[-1].date()}")

Loaded 4 timeframes: ['1wk', '1d', '4h', '1h']
  1wk     104 rows  2024-06-09 → 2026-05-31
  1d      581 rows  2024-06-06 → 2026-06-05
  4h     3174 rows  2024-06-06 → 2026-06-05
  1h    12259 rows  2024-06-05 → 2026-06-05


## 2. Detect bases, then classify formations

In [2]:
# Run base detection + formation classification on all timeframes
results = {}
for tf, df in data.items():
    passed, _ = detect_bases(df)
    formations = detect_formations(df, passed)
    results[tf] = {
        "df":         df,
        "passed":     passed,
        "formations": formations,
    }

# Summary table
rows = []
for tf, r in results.items():
    fms = r["formations"]
    n_demand = sum(1 for f in fms if f["zone_type"] == "demand")
    n_supply = sum(1 for f in fms if f["zone_type"] == "supply")
    n_bases  = len(r["passed"])
    n_skipped = n_bases - len(fms)
    rows.append({
        "timeframe":          tf,
        "passed_bases":       n_bases,
        "formations_found":   len(fms),
        "demand_zones":       n_demand,
        "supply_zones":       n_supply,
        "skipped (flat leg)": n_skipped,
        "form_rate":          f"{len(fms)/n_bases:.0%}" if n_bases else "—",
    })

summary = pd.DataFrame(rows).set_index("timeframe")
display(summary)

,passed_bases,formations_found,demand_zones,supply_zones,skipped (flat leg),form_rate
timeframe,,,,,,
1wk,25,18,12,6,7,72%
1d,157,131,71,60,26,83%
4h,812,637,343,294,175,78%
1h,3113,2357,1251,1106,756,76%


## 3. Inspect — daily formations

In [3]:
TF = "1d"
df_1d = results[TF]["df"]

detail_rows = []
for f in results[TF]["formations"]:
    bs, be = f["start"], f["end"]
    detail_rows.append({
        "date_start":   df_1d.index[bs].date(),
        "date_end":     df_1d.index[be].date(),
        "base_candles": f["count"],
        "formation":    f["formation"],
        "zone_type":    f["zone_type"],
        "leg_in_dir":   f["leg_in_dir"],
        "leg_out_dir":  f["leg_out_dir"],
        "leg_in_up":    f["leg_in_up"],
        "leg_in_down":  f["leg_in_down"],
        "leg_out_up":   f["leg_out_up"],
        "leg_out_down": f["leg_out_down"],
        "leg_strength": f["leg_strength"],
        "avg_atr":      f["avg_atr"],
    })

detail = pd.DataFrame(detail_rows)
if detail.empty:
    print(f"No formations found for {TF}")
else:
    display(detail.to_string(index=False))

'date_start   date_end  base_candles formation zone_type leg_in_dir leg_out_dir  leg_in_up  leg_in_down  leg_out_up  leg_out_down  leg_strength  avg_atr\n2024-06-16 2024-06-19             4       RBR    demand         up          up    1.80699      0.74001     1.94800       0.00700         0.981  0.58287\n2024-06-24 2024-06-25             2       RBR    demand         up          up    1.94400      0.01100     1.58299       0.03700         0.772  0.66438\n2024-06-27 2024-06-28             2       RBR    demand         up          up    1.88699      0.23801     0.90799       0.10400         0.651  0.70092\n2024-07-08 2024-07-08             1       DBD    supply       down        down    0.31200      0.96300     0.94600       3.45000         0.762  0.71447\n2024-07-12 2024-07-16             4       DBD    supply       down        down    0.86400      3.53200     0.26500       2.99500         0.823  1.01838\n2024-07-19 2024-07-19             1       RBD    supply         up        down   

## 4. Visualize — all 4 timeframes

In [4]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import timedelta

PLOT_MONTHS = 2

# Color by zone type
FILL = {
    "demand": "rgba(38,166,154,0.18)",
    "supply": "rgba(239,83,80,0.18)",
}
LINE_COLOR = {
    "demand": "#26a69a",
    "supply": "#ef5350",
}

def plot_formations(tf: str, r: dict) -> None:
    df          = r["df"]
    formations  = r["formations"]

    # Slice to last PLOT_MONTHS
    cutoff   = df.index[-1] - pd.DateOffset(months=PLOT_MONTHS)
    df_plot  = df[df.index >= cutoff].copy()
    plot_start = df_plot.index[0]
    offset   = len(df) - len(df_plot)

    # Only show formations whose cluster end is inside the window
    vis_fms = [f for f in formations if f["end"] >= offset]

    n_demand_all = sum(1 for f in formations if f["zone_type"] == "demand")
    n_supply_all = sum(1 for f in formations if f["zone_type"] == "supply")

    fig = go.Figure()

    # Candlestick
    fig.add_trace(go.Candlestick(
        x=df_plot.index,
        open=df_plot["open"],
        high=df_plot["high"],
        low=df_plot["low"],
        close=df_plot["close"],
        increasing_line_color=COLOR_BULL,
        decreasing_line_color=COLOR_BEAR,
        name="Price",
    ))

    # Shade each formation's base cluster
    for f in vis_fms:
        bs_plot = f["start"] - offset
        be_plot = f["end"]   - offset

        # Clamp to the visible window
        bs_plot = max(bs_plot, 0)
        be_plot = min(be_plot, len(df_plot) - 1)

        x0 = df_plot.index[bs_plot]
        x1 = df_plot.index[be_plot]
        zt = f["zone_type"]

        # Vertical region for the base
        fig.add_vrect(
            x0=x0, x1=x1,
            fillcolor=FILL[zt],
            line_color=LINE_COLOR[zt],
            line_width=1,
            opacity=1.0,
        )

        # Formation label centered on the cluster
        mid_pos = (bs_plot + be_plot) // 2
        mid_x   = df_plot.index[mid_pos]
        mid_y   = df_plot["high"].iloc[bs_plot : be_plot + 1].max()
        fig.add_annotation(
            x=mid_x, y=mid_y,
            text=f["formation"],
            showarrow=False,
            font=dict(size=10, color=LINE_COLOR[zt]),
            yanchor="bottom",
        )

    fig.update_layout(
        title=(
            f"{SYMBOL} {tf} — Formations in last {PLOT_MONTHS} months "
            f"(shown: {len(vis_fms)} | full history: "
            f"{n_demand_all}D + {n_supply_all}S = {len(formations)} total)"
        ),
        xaxis_rangeslider_visible=False,
        template="plotly_dark",
        paper_bgcolor=BG,
        plot_bgcolor=BG,
        xaxis=dict(gridcolor=GRID),
        yaxis=dict(gridcolor=GRID),
        height=480,
    )
    fig.show()


for tf, r in results.items():
    plot_formations(tf, r)

## 5. What's next

| Notebook | Topic | Status |
|---|---|---|
| `06_proximal_distal_departure.ipynb` | Zone boundaries (proximal/distal) + departure strength | ⏳ next |
| `07_verify_all_scenarios.ipynb` | End-to-end pipeline — all filters together | ⏳ |

With formations classified we now know:
- **Which side** of the market each zone represents (demand vs supply)
- **How strong** the move was that created and left the base

Next we define the precise **entry / reaction levels** (proximal line, distal line) and verify that price **departed** fast enough to confirm institutional intent.